# 03 — Feature Extraction
**Goal:** Load every audio file from `dataset.csv`, extract MFCC features using librosa, and save the results to `features.csv` for model training.

## 1. Check Libraries

In [1]:
import librosa
import numpy as np
import pandas as pd
import tqdm
import sklearn
import warnings
warnings.filterwarnings('ignore')

print(f"librosa:      {librosa.__version__}")
print(f"numpy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")
print(f"tqdm:         {tqdm.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print("\nAll libraries ready ✅")

librosa:      0.11.0
numpy:        2.3.5
pandas:       2.3.3
tqdm:         4.67.1
scikit-learn: 1.7.2

All libraries ready ✅


## 2. Load dataset.csv

In [2]:
df = pd.read_csv('dataset.csv')
print(f"Total files to process: {len(df)}")
print(f"Emotions: {df['emotion'].unique().tolist()}")
df.head()

Total files to process: 4240
Emotions: ['neutral', 'happy', 'sad', 'angry', 'fear', 'disgust', 'surprise']


,path,emotion,source
0,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
1,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
2,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
3,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS
4,D:\Projects\CSE445\speech_emotion_recognation\...,neutral,RAVDESS


## 3. Feature Extraction Settings

In [3]:
# Audio settings
SAMPLE_RATE  = 22050   # standard librosa sample rate
DURATION     = 3       # seconds — crop or pad all audio to 3 seconds
N_MFCC       = 40      # number of MFCC coefficients to extract

# Emotion label encoding
EMOTION_LABELS = ['neutral', 'happy', 'sad', 'angry', 'fear', 'disgust', 'surprise']
LABEL_MAP = {emotion: idx for idx, emotion in enumerate(EMOTION_LABELS)}

print(f"Sample rate:  {SAMPLE_RATE} Hz")
print(f"Duration:     {DURATION} seconds")
print(f"MFCC coeffs:  {N_MFCC}")
print(f"Label map:    {LABEL_MAP}")

Sample rate:  22050 Hz
Duration:     3 seconds
MFCC coeffs:  40
Label map:    {'neutral': 0, 'happy': 1, 'sad': 2, 'angry': 3, 'fear': 4, 'disgust': 5, 'surprise': 6}


## 4. Feature Extraction Function

In [4]:
def extract_features(file_path, sample_rate=SAMPLE_RATE, duration=DURATION, n_mfcc=N_MFCC):
    """
    Loads an audio file, pads/crops to fixed duration,
    extracts 40 MFCC coefficients, and returns their mean values.
    Returns a numpy array of shape (40,) or None if file fails.
    """
    try:
        # Load audio — fixed duration, mono channel
        y, sr = librosa.load(file_path, sr=sample_rate, duration=duration, mono=True)

        # Pad with zeros if audio is shorter than duration
        target_length = sample_rate * duration
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)))

        # Extract MFCC — shape: (n_mfcc, time_frames)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

        # Take mean across time axis — shape: (n_mfcc,)
        mfcc_mean = np.mean(mfcc, axis=1)

        return mfcc_mean

    except Exception as e:
        print(f"  Error processing {file_path}: {e}")
        return None

print("Feature extraction function defined.")

# Quick test on one file
test_path = df.iloc[0]['path']
test_features = extract_features(test_path)
print(f"Test file: {test_path}")
print(f"Feature shape: {test_features.shape}")
print(f"Feature values (first 5): {test_features[:5].round(3)}")

Feature extraction function defined.
Test file: D:\Projects\CSE445\speech_emotion_recognation\data\RAVDESS\Actor_01\03-01-01-01-01-01-01.wav
Feature shape: (40,)
Feature values (first 5): [-6.81925e+02  6.02540e+01  6.06000e-01  1.35580e+01  8.39000e+00]


## 5. Extract Features for All Files
⚠️ This cell processes 4,240 audio files — may take **10–20 minutes**. Do not close the notebook.

In [5]:
from tqdm import tqdm

features_list = []
labels_list   = []
skipped       = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
    features = extract_features(row['path'])
    if features is not None:
        features_list.append(features)
        labels_list.append(LABEL_MAP[row['emotion']])
    else:
        skipped += 1

print(f"\nDone!")
print(f"Successfully extracted: {len(features_list)} files")
print(f"Skipped (errors):       {skipped} files")

Extracting features: 100%|█████████████████████████████████████████████████████████| 4240/4240 [01:17<00:00, 54.73it/s]


Done!
Successfully extracted: 4240 files
Skipped (errors):       0 files


## 6. Build Features DataFrame

In [7]:
# Each row = one audio file
# Columns = mfcc_0, mfcc_1, ..., mfcc_39, label
feature_columns = [f'mfcc_{i}' for i in range(N_MFCC)]

features_df = pd.DataFrame(features_list, columns=feature_columns)
features_df['label'] = labels_list

print(f"Features DataFrame shape: {features_df.shape}")
print(f"Columns: {list(features_df.columns[:5])} ... label")
print(f"\nLabel distribution:")
print(features_df['label'].value_counts().sort_index().to_string())
features_df.head()

Features DataFrame shape: (4240, 41)
Columns: ['mfcc_0', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4'] ... label

Label distribution:
label
0    688
1    592
2    592
3    592
4    592
5    592
6    592


,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,...,mfcc_31,mfcc_32,mfcc_33,mfcc_34,mfcc_35,mfcc_36,mfcc_37,mfcc_38,mfcc_39,label
0,-681.925293,60.253666,0.606010,13.558146,8.389651,0.470501,-3.646512,-3.577835,-12.171618,-3.223377,...,-2.347052,-2.998035,-2.032508,-3.536474,-1.709354,-1.207591,-1.952652,-3.854637,-1.693044,0
1,-674.354797,61.282314,-1.758771,17.722033,9.725379,-0.205105,-1.563526,-5.904860,-12.915890,-1.533724,...,-2.980965,-2.388265,-2.739164,-3.356074,-0.902344,-1.438924,-3.428027,-3.562944,-2.562083,0
2,-677.840332,62.683060,-0.074478,14.536501,5.596521,1.039011,-2.469042,-6.033867,-11.388467,-4.138160,...,-2.332745,-2.617405,-2.085288,-3.020938,-2.170061,-1.221695,-2.983992,-3.828454,-2.457575,0
3,-676.639771,58.588737,2.639020,13.680299,7.092484,2.882377,-1.914979,-7.563103,-11.114905,-3.638864,...,-2.318527,-3.073290,-2.957253,-3.449866,-1.575281,-1.043560,-2.619672,-3.638883,-3.580294,0
4,-695.960327,72.433807,2.809904,16.982683,8.941421,1.745872,-4.281387,-4.778624,-11.778305,-5.999314,...,-1.476523,-2.303163,-2.894165,-2.110953,-0.509307,-2.114303,-2.019947,-4.525523,-3.172242,0


## 7. Check for Missing Values

In [8]:
missing = features_df.isnull().sum().sum()
if missing == 0:
    print("No missing values found ✅")
else:
    print(f"WARNING: {missing} missing values found — dropping those rows")
    features_df = features_df.dropna().reset_index(drop=True)
    print(f"Remaining rows: {len(features_df)}")

No missing values found ✅


## 8. Save features.csv

In [11]:
features_df.to_csv('features.csv', index=False)
print(f"Saved: features.csv")
print(f"Shape: {features_df.shape}")
print(f"\n→ 4240 rows × 41 columns (40 MFCC features + 1 label)")

Saved: features.csv
Shape: (4240, 41)

→ 4240 rows × 41 columns (40 MFCC features + 1 label)


---
**Done!** `features.csv` is ready. Next step → `04_model_training.ipynb`